# EGFR "Virtual Biopsy" — v3: **ViT over CT slices**
### CT + clinical -> EGFR mutation (Mutant vs Wildtype)

**What changed from v2.** The image branch is now a *transformer over slices* (`IMAGE_BACKBONE="vit_slices"`).
The whole CT cube is fed in as a **sequence of slices**: every slice becomes **one token**, a small
Transformer reasons across the slices, and a CLS token is pooled into the image feature vector.

**How this maps to "a ViT whose patches are slices".**
In a normal ViT the *patch-embedding* is a conv that turns a 16x16 patch into a token. Here the
patch-embedding is a **pretrained 2D ResNet18** that turns a whole slice into a token. After that it is the
same ViT machinery — CLS token, positional embeddings, multi-head attention across tokens. So each token
*is* one slice, exactly the idea, but each slice is embedded by something that preserves its 2D content
instead of being flattened (flattening a whole slice into one linear token throws away the in-slice spatial
structure and cannot reuse ImageNet weights — see the chat notes).

**Honest caveat (read the project notes).** At N=111 / ~21 positives the bottleneck is statistical, not
architectural. This is a bigger-capacity model, so it is prone to overfit. The per-slice backbone is
**frozen by default** to keep the trainable parameter count small, and an optional **repeated CV**
(`N_REPEATS`) is included so you measure mean+/-std rather than one noisy AUROC. Treat this as an
experiment; do not be surprised if it ties or trails the 2.5D ResNet. The AIM-recovery + repeated-CV data
steps remain the higher-value move.


## 1. Install dependencies (run once)

In [1]:
import sys
# !{sys.executable} -m pip install -q "pydicom<3" pydicom-seg SimpleITK scipy scikit-learn matplotlib tqdm torchvision
print("done")
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

done


## 2. Imports

In [2]:
import os, glob, shutil, random, warnings, numpy as np, pandas as pd
from pathlib import Path
import pydicom, pydicom_seg, SimpleITK as sitk
from scipy import ndimage
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch", torch.__version__, "| Device:", DEVICE)

d:\git\PhDProjects\LungCancerGrading\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch 2.12.1+cu126 | Device: cuda


## 3. CONFIG — all knobs here

In [3]:
# ---------- PATHS (EDIT) ----------
DATA_ROOT    = Path("data/nsclc_radiogenomics")
CLINICAL_CSV = Path("data/NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv")
AIM_DIR      = Path("data/AIM_files_updated-11-10-2020/AIM_files_updated-11-10-2020")   # optional baseline cell
CACHE_BASE   = Path("egfr_cache")
OUTPUT_DIR   = Path("egfr_out"); OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- TARGET ----------
TARGET_COL, POS_LABEL, NEG_LABEL = "EGFR mutation status", "Mutant", "Wildtype"

# ---------- IMAGE PREPROCESSING ----------
TARGET_SHAPE   = (32, 64, 64)     # (D,H,W) cached cube  -> D=32 slices become the token sequence
CROP_PAD_VOX   = 16
HU_MIN, HU_MAX = -1000, 400
REQUIRE_SEG    = True
ALLOW_CENTER_CROP_FALLBACK = False
FORCE_REBUILD  = False            # cubes are identical to v2; reuse the cache. Set True only if you change preprocessing.

# ---------- TABULAR ----------
INCLUDE_HISTOLOGY = False
INCLUDE_ETHNICITY = True

# ---------- MODEL ----------
IMAGE_BACKBONE = "vit_slices"     # "vit_slices" (NEW) | "2.5d" (pretrained ResNet18, 3 slices) | "3d" (small CNN)
PRETRAINED     = True
IMG_FEAT_DIM   = 64
TAB_HIDDEN     = 64
FUSION_HIDDEN  = 64
DROPOUT        = 0.4

# ---- slice-transformer (ViT-over-slices) knobs ----
FREEZE_SLICE_BACKBONE = True      # keep the per-slice CNN frozen. ESSENTIAL at this N (bounds trainable params)
VIT_D_MODEL  = 192                # token width of the cross-slice transformer
VIT_HEADS    = 3
VIT_LAYERS   = 2
VIT_FF       = 384
SLICE_RES    = 224                # each slice resized to this before the 2D backbone

# ---------- TRAINING ----------
N_FOLDS = 5
N_REPEATS = 1                     # set to 3+ for a trustworthy mean+/-std (slower). At this N you NEED >1 to compare models.
EPOCHS  = 40                      # the cross-slice transformer converges faster than v2's 3D CNN
LR      = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 8                  # 8 cubes x 32 slices = 256 frozen ResNet18 forwards/step (fits 8GB with AMP)
USE_AUGMENTATION  = True
USE_CLASS_WEIGHTS = True
SEED = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PRE_TAG = f"seg{int(REQUIRE_SEG)}_{TARGET_SHAPE[0]}x{TARGET_SHAPE[1]}x{TARGET_SHAPE[2]}_pad{CROP_PAD_VOX}"
CACHE_DIR = CACHE_BASE / PRE_TAG
if FORCE_REBUILD and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("cache:", CACHE_DIR)

cache: egfr_cache\seg1_32x64x64_pad16


## 4. Clinical labels + tabular features (incl. ethnicity)

In [4]:
clin = pd.read_csv(CLINICAL_CSV)
clin["Case ID"] = clin["Case ID"].astype(str).str.strip()
clin["label"] = clin[TARGET_COL].map({POS_LABEL: 1, NEG_LABEL: 0})
clin = clin.dropna(subset=["label"]).copy(); clin["label"] = clin["label"].astype(int)
print("Usable EGFR labels:", len(clin)); print(clin["label"].value_counts().rename({0:NEG_LABEL,1:POS_LABEL}))

def build_tabular(df):
    f = pd.DataFrame(index=df.index)
    age = pd.to_numeric(df["Age at Histological Diagnosis"], errors="coerce")
    f["age"] = age.fillna(age.median())/100.0
    f["male"] = (df["Gender"].astype(str).str.strip().str.lower()=="male").astype(float)
    sm = df["Smoking status"].astype(str).str.strip()
    for v in ["Nonsmoker","Former","Current"]: f[f"smk_{v}"] = (sm==v).astype(float)
    for c in [c for c in df.columns if c.startswith("Tumor Location")]:
        key = c.split("choice=")[-1].rstrip(")").strip().replace(" ","_")
        f[f"loc_{key}"] = (df[c].astype(str).str.strip().str.lower()=="checked").astype(float)
    f = pd.concat([f, pd.get_dummies(df["%GG"].astype(str).str.strip(), prefix="gg").astype(float)], axis=1)
    if INCLUDE_ETHNICITY and "Ethnicity" in df.columns:
        f = pd.concat([f, pd.get_dummies(df["Ethnicity"].astype(str).str.strip(), prefix="eth").astype(float)], axis=1)
    if INCLUDE_HISTOLOGY:
        f = pd.concat([f, pd.get_dummies(df["Histology "].astype(str).str.strip(), prefix="hist").astype(float)], axis=1)
    return f

print("Tabular features:", build_tabular(clin).shape[1])

Usable EGFR labels: 172
label
Wildtype    129
Mutant       43
Name: count, dtype: int64
Tabular features: 25


## 5. Index DICOM images (CT + segmentation)

In [5]:
def scan_patient(pdir):
    info = {"ct_dir": None, "ct_n": 0, "seg_file": None, "seg_modality": None}
    for root, _, files in os.walk(pdir):
        dcms = [f for f in files if f.lower().endswith(".dcm")]
        if not dcms: continue
        try:
            ds = pydicom.dcmread(os.path.join(root, dcms[0]), stop_before_pixels=True, force=True)
        except Exception: continue
        mod = str(getattr(ds, "Modality", "")).upper()
        if mod == "CT":
            if len(dcms) > info["ct_n"]: info["ct_dir"], info["ct_n"] = root, len(dcms)
        elif mod in ("SEG","RTSTRUCT"):
            info["seg_file"], info["seg_modality"] = os.path.join(root, dcms[0]), mod
    return info

records = []
for cid in tqdm(clin["Case ID"].tolist(), desc="indexing"):
    pdir = DATA_ROOT/cid
    info = scan_patient(pdir) if pdir.exists() else {"ct_dir":None,"ct_n":0,"seg_file":None,"seg_modality":None}
    info["Case ID"] = cid; records.append(info)
idx = pd.DataFrame(records)
cohort = clin.merge(idx, on="Case ID", how="left")
print("CT:", cohort["ct_dir"].notna().sum(), "| SEG:", (cohort["seg_modality"]=="SEG").sum(),
      "| RTSTRUCT:", (cohort["seg_modality"]=="RTSTRUCT").sum(), "| neither:", cohort["seg_modality"].isna().sum())

indexing: 100%|██████████| 172/172 [00:25<00:00,  6.84it/s]

CT: 172 | SEG: 117 | RTSTRUCT: 0 | neither: 55


## 6. Preprocess -> tumour cubes (reuses the v2 cache unless FORCE_REBUILD)

In [6]:
def load_ct(ct_dir):
    r = sitk.ImageSeriesReader(); r.SetFileNames(r.GetGDCMSeriesFileNames(str(ct_dir))); return r.Execute()

def seg_mask_on_ct(seg_file, ct_img):
    result = pydicom_seg.SegmentReader().read(pydicom.dcmread(seg_file))
    mask = None
    for num in result.available_segments:
        res = sitk.Resample(result.segment_image(num), ct_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
        a = sitk.GetArrayFromImage(res).astype(bool)
        mask = a if mask is None else (mask | a)
    return mask

def crop_to_cube(ct_arr, mask):
    if mask is not None and mask.any():
        zz,yy,xx = np.where(mask); p=CROP_PAD_VOX
        z0,z1=max(0,zz.min()-p),min(ct_arr.shape[0],zz.max()+p+1)
        y0,y1=max(0,yy.min()-p),min(ct_arr.shape[1],yy.max()+p+1)
        x0,x1=max(0,xx.min()-p),min(ct_arr.shape[2],xx.max()+p+1)
    else:
        Z,Y,X=ct_arr.shape; z0,z1=int(Z*.25),int(Z*.75);y0,y1=int(Y*.2),int(Y*.8);x0,x1=int(X*.2),int(X*.8)
    crop = ct_arr[z0:z1,y0:y1,x0:x1].astype(np.float32)
    crop = ndimage.zoom(crop, [t/max(1,s) for t,s in zip(TARGET_SHAPE,crop.shape)], order=1)
    crop = np.clip(crop, HU_MIN, HU_MAX)
    return ((crop-HU_MIN)/(HU_MAX-HU_MIN)).astype(np.float32)

status = []
for _, row in tqdm(cohort.iterrows(), total=len(cohort), desc="preprocess"):
    cid = row["Case ID"]; out = CACHE_DIR/f"{cid}.npy"
    if out.exists(): status.append((cid,"cached")); continue
    if pd.isna(row["ct_dir"]): status.append((cid,"no_ct")); continue
    try:
        ct = load_ct(row["ct_dir"]); ct_arr = sitk.GetArrayFromImage(ct)
        mask = None
        if row["seg_modality"]=="SEG":
            try: mask = seg_mask_on_ct(row["seg_file"], ct)
            except Exception: mask = None
        if mask is None or not mask.any():
            if REQUIRE_SEG or not ALLOW_CENTER_CROP_FALLBACK:
                status.append((cid,"no_seg_skipped")); continue
            np.save(out, crop_to_cube(ct_arr, None)); status.append((cid,"ok_centercrop")); continue
        np.save(out, crop_to_cube(ct_arr, mask)); status.append((cid,"ok_seg"))
    except Exception as e:
        status.append((cid, f"error:{type(e).__name__}"))

st = pd.DataFrame(status, columns=["Case ID","status"]); print(st["status"].value_counts())
cohort = cohort.merge(st, on="Case ID", how="left")
cohort = cohort[cohort["status"].isin(["ok_seg","ok_centercrop","cached"])].reset_index(drop=True)
print("\nFinal cohort:", len(cohort)); print(cohort["label"].value_counts().rename({0:NEG_LABEL,1:POS_LABEL}))

preprocess: 100%|██████████| 172/172 [03:42<00:00,  1.29s/it]

status
cached            111
no_seg_skipped     61
Name: count, dtype: int64

Final cohort: 111
label
Wildtype    90
Mutant      21
Name: count, dtype: int64


## 7. Tabular matrix + Dataset (train-time augmentation)

In [7]:
tab_mat = build_tabular(cohort).values.astype(np.float32)
N_TAB = tab_mat.shape[1]; labels = cohort["label"].values.astype(np.float32)
print("Tabular dim:", N_TAB)

def augment_cube(cube):                       # cube: torch [1,D,H,W] in [0,1]
    if random.random()<0.5: cube = torch.flip(cube, dims=[2])   # flip H
    if random.random()<0.5: cube = torch.flip(cube, dims=[3])   # flip W
    cube = cube*random.uniform(0.9,1.1) + random.uniform(-0.05,0.05)   # intensity jitter
    return cube.clamp(0,1)

class EGFRDataset(Dataset):
    def __init__(self, indices, train=False):
        self.indices=list(indices); self.train=train
    def __len__(self): return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        cube = np.load(CACHE_DIR/f"{cohort.iloc[i]['Case ID']}.npy")
        img = torch.from_numpy(cube).float().unsqueeze(0)        # [1,D,H,W]
        if self.train and USE_AUGMENTATION: img = augment_cube(img)
        tab = torch.from_numpy(tab_mat[i]).float()
        y = torch.tensor([labels[i]], dtype=torch.float32)
        return img, tab, y

Tabular dim: 23


## 8. Model — **ViT over slices** (+ the v2 backbones kept as switches)

**The new branch (`SliceTransformerEncoder`).** Each of the D=32 slices in the tumour cube is pushed
through a pretrained 2D ResNet18 to produce one 512-d vector, projected to a token. We prepend a learnable
CLS token, add learnable positional embeddings (so the transformer knows slice order), and run a small
Transformer encoder so attention can relate slices to each other (e.g. how the tumour changes through its
depth). The CLS output is the image feature vector. The ResNet18 is **frozen** so only the projection +
transformer + CLS/pos are trained — a few hundred-k params, appropriate for ~110 cubes.

Set `IMAGE_BACKBONE` back to `"2.5d"` or `"3d"` to recover the v2 branches.

In [8]:
class SliceTransformerEncoder(nn.Module):
    """'ViT over CT slices': the whole cube is a sequence of slices; each slice is ONE token
    (embedded by a pretrained 2D backbone = the patch-embedding analogue), a small Transformer
    reasons ACROSS slices, and a CLS token is pooled to the image feature vector."""
    def __init__(self, out_dim=IMG_FEAT_DIM, pretrained=PRETRAINED,
                 freeze_backbone=FREEZE_SLICE_BACKBONE, d_model=VIT_D_MODEL,
                 n_heads=VIT_HEADS, n_layers=VIT_LAYERS, ff=VIT_FF,
                 p=DROPOUT, res=SLICE_RES, max_slices=64):
        super().__init__()
        self.freeze_backbone = freeze_backbone; self.res = res
        weights = tvm.ResNet18_Weights.DEFAULT if pretrained else None
        net = tvm.resnet18(weights=weights)
        self.slice_backbone = nn.Sequential(*list(net.children())[:-1])    # -> [N,512,1,1]
        self.backbone_out = 512
        if freeze_backbone:
            for prm in self.slice_backbone.parameters(): prm.requires_grad = False
        self.proj_in = nn.Linear(self.backbone_out, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, max_slices + 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=ff,
                                           dropout=p, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.proj_out = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(p),
                                      nn.Linear(d_model, out_dim), nn.ReLU(inplace=True))
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        nn.init.trunc_normal_(self.cls, std=0.02); nn.init.trunc_normal_(self.pos, std=0.02)

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_backbone: self.slice_backbone.eval()   # keep frozen BN on running stats
        return self

    def _embed_slices(self, x):
        if self.freeze_backbone:
            with torch.no_grad(): return self.slice_backbone(x)
        return self.slice_backbone(x)

    def forward(self, cube):                          # cube: [B,1,D,H,W] in [0,1]
        B, _, D, H, W = cube.shape
        x = cube[:,0].reshape(B*D, 1, H, W)           # every slice -> its own image
        x = x.repeat(1, 3, 1, 1)                       # 1ch -> 3ch
        x = F.interpolate(x, size=(self.res, self.res), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std                # ImageNet normalisation
        feat = self._embed_slices(x).flatten(1)       # [B*D, 512]
        tok = self.proj_in(feat).reshape(B, D, -1)    # [B, D, d_model]  (one token per slice)
        cls = self.cls.expand(B, -1, -1)              # [B, 1, d_model]
        seq = torch.cat([cls, tok], dim=1)            # [B, D+1, d_model]
        seq = seq + self.pos[:, :seq.shape[1], :]
        out = self.transformer(seq)                   # attention across slices
        return self.proj_out(out[:, 0])               # CLS -> [B, out_dim]

class Image25DEncoder(nn.Module):
    def __init__(self, out_dim=IMG_FEAT_DIM, pretrained=PRETRAINED, p=DROPOUT):
        super().__init__()
        weights = tvm.ResNet18_Weights.DEFAULT if pretrained else None
        net = tvm.resnet18(weights=weights)
        self.features = nn.Sequential(*list(net.children())[:-1])
        self.proj = nn.Sequential(nn.Flatten(), nn.Dropout(p), nn.Linear(512, out_dim), nn.ReLU(inplace=True))
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def forward(self, cube):
        D = cube.shape[2]; idx = [D//4, D//2, (3*D)//4]
        x = cube[:,0][:, idx, :, :]
        x = F.interpolate(x, size=(224,224), mode="bilinear", align_corners=False)
        x = (x - self.mean)/self.std
        return self.proj(self.features(x))

class Image3DEncoder(nn.Module):
    def __init__(self, out_dim=IMG_FEAT_DIM, p=DROPOUT):
        super().__init__()
        def blk(ci,co): return nn.Sequential(nn.Conv3d(ci,co,3,padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True), nn.MaxPool3d(2))
        self.net = nn.Sequential(blk(1,16), blk(16,32), blk(32,64), nn.AdaptiveAvgPool3d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Dropout(p), nn.Linear(64,out_dim), nn.ReLU(inplace=True))
    def forward(self,x): return self.proj(self.net(x))

class TabularEncoder(nn.Module):
    def __init__(self, n_in, hidden=TAB_HIDDEN, out_dim=32, p=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in,hidden), nn.ReLU(inplace=True), nn.Dropout(p),
                                 nn.Linear(hidden,out_dim), nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)

def make_image_encoder():
    if IMAGE_BACKBONE == "vit_slices": return SliceTransformerEncoder()
    if IMAGE_BACKBONE == "2.5d":       return Image25DEncoder()
    return Image3DEncoder()

class FusionNet(nn.Module):
    def __init__(self, n_tab, use_image=True, use_tabular=True):
        super().__init__(); self.use_image, self.use_tabular = use_image, use_tabular; dim=0
        if use_image:
            self.img = make_image_encoder(); dim += IMG_FEAT_DIM
        if use_tabular:
            self.tab = TabularEncoder(n_tab); dim += 32
        self.head = nn.Sequential(nn.Linear(dim,FUSION_HIDDEN), nn.ReLU(inplace=True),
                                  nn.Dropout(DROPOUT), nn.Linear(FUSION_HIDDEN,1))
    def forward(self, img, tab):
        f=[]
        if self.use_image:   f.append(self.img(img))
        if self.use_tabular: f.append(self.tab(tab))
        return self.head(torch.cat(f,1))

_m = FusionNet(N_TAB)
print("Backbone:", IMAGE_BACKBONE)
print("FusionNet total params:    ", sum(p.numel() for p in _m.parameters()))
print("FusionNet trainable params:", sum(p.numel() for p in _m.parameters() if p.requires_grad))
del _m

Backbone: vit_slices
FusionNet total params:     11904353
FusionNet trainable params: 727841


## 9. Train + predict (cosine LR, AMP, only trainable params optimised)

In [9]:
from torch.cuda.amp import autocast, GradScaler

@torch.no_grad()
def predict(model, loader):
    model.eval(); P,Y=[],[]
    for img,tab,y in loader:
        img,tab = img.to(DEVICE), tab.to(DEVICE)
        with autocast(enabled=(DEVICE=="cuda")): logit = model(img,tab)
        P.append(torch.sigmoid(logit).float().cpu().numpy().ravel()); Y.append(y.numpy().ravel())
    return np.concatenate(P), np.concatenate(Y)

def train_one(train_idx, val_idx, use_image=True, use_tabular=True):
    tr = DataLoader(EGFRDataset(train_idx, train=True),  batch_size=BATCH_SIZE, shuffle=True)
    va = DataLoader(EGFRDataset(val_idx,   train=False), batch_size=BATCH_SIZE, shuffle=False)
    model = FusionNet(N_TAB, use_image, use_tabular).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]   # skip the frozen backbone
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = GradScaler(enabled=(DEVICE=="cuda"))
    pw = None
    if USE_CLASS_WEIGHTS:
        n_pos = labels[train_idx].sum(); pw = torch.tensor([(len(train_idx)-n_pos)/max(1.0,n_pos)], device=DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    best_auc, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for img,tab,y in tr:
            img,tab,y = img.to(DEVICE), tab.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with autocast(enabled=(DEVICE=="cuda")): loss = crit(model(img,tab), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        p,yt = predict(model, va); auc = roc_auc_score(yt,p) if len(np.unique(yt))>1 else 0.5
        if auc>best_auc: best_auc=auc; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state); return model

## 10. Cross-validation (with optional repeats) + evaluation

`run_cv_repeated` re-runs the whole 5-fold CV over `N_REPEATS` different shuffles and reports the
mean+/-std of all fold AUROCs. **At ~21 positives a single number is noise** (the project's own finding) —
bump `N_REPEATS` to 3-5 before believing any ranking between models.

In [10]:
def run_cv(use_image=True, use_tabular=True, tag="fusion", seed=SEED):
    strat = (cohort["label"].astype(str)+"_"+cohort["Patient affiliation"].astype(str)).values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros(len(cohort)); fa=[]
    for f,(tr,va) in enumerate(skf.split(np.arange(len(cohort)), strat)):
        m = train_one(tr, va, use_image, use_tabular)
        p,yt = predict(m, DataLoader(EGFRDataset(va), batch_size=BATCH_SIZE)); oof[va]=p
        a = roc_auc_score(yt,p) if len(np.unique(yt))>1 else 0.5; fa.append(a)
        print(f"[{tag}] seed {seed} fold {f} AUROC {a:.3f}")
    return oof, fa

def run_cv_repeated(use_image=True, use_tabular=True, tag="fusion", n_repeats=N_REPEATS):
    all_fa=[]; oof_sum=np.zeros(len(cohort))
    for r in range(n_repeats):
        oof, fa = run_cv(use_image, use_tabular, tag, seed=SEED+r)
        all_fa += fa; oof_sum += oof
        print(f"[{tag}] seed {SEED+r} mean fold AUROC {np.mean(fa):.3f}")
    oof_mean = oof_sum/n_repeats
    print(f"[{tag}] OVERALL {n_repeats}x{N_FOLDS} folds: {np.mean(all_fa):.3f} +/- {np.std(all_fa):.3f}\n")
    return oof_mean, all_fa

def evaluate(oof, name):
    y = labels.astype(int)
    auroc, auprc = roc_auc_score(y,oof), average_precision_score(y,oof)
    fpr,tpr,thr = roc_curve(y,oof); t = thr[np.argmax(tpr-fpr)]
    tn,fp,fn,tp = confusion_matrix(y,(oof>=t).astype(int)).ravel()
    print(f"=== {name} ===\n  AUROC {auroc:.3f} | AUPRC {auprc:.3f} (prevalence {y.mean():.2f})")
    print(f"  @thr {t:.2f}: sensitivity {tp/(tp+fn):.2f} specificity {tn/(tn+fp):.2f}")
    print(f"  confusion [[TN {tn}, FP {fp}],[FN {fn}, TP {tp}]]")
    return dict(auroc=auroc, auprc=auprc)

oof_fusion, _ = run_cv_repeated(True, True, "fusion")
_ = evaluate(oof_fusion, "FUSION (CT + clinical)")

[fusion] seed 0 fold 0 AUROC 0.528
[fusion] seed 0 fold 1 AUROC 0.646
[fusion] seed 0 fold 2 AUROC 0.319
[fusion] seed 0 fold 3 AUROC 0.396
[fusion] seed 0 fold 4 AUROC 0.771
[fusion] seed 0 mean fold AUROC 0.532
[fusion] OVERALL 1x5 folds: 0.532 +/- 0.164

=== FUSION (CT + clinical) ===
  AUROC 0.473 | AUPRC 0.181 (prevalence 0.19)
  @thr 0.50: sensitivity 0.43 specificity 0.64
  confusion [[TN 58, FP 32],[FN 12, TP 9]]


## 11. Ablation — fusion vs image-only vs clinical-only

The decisive comparison. Watch whether the **image-only** slice-transformer rises above v2's image-only
(~0.64-0.66) and whether **fusion** clears clinical-only. Compare the **mean+/-std**, not single folds.

In [11]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
oof_lr = cross_val_predict(lr, tab_mat, labels.astype(int), cv=N_FOLDS, method="predict_proba")[:,1]
oof_img,  _ = run_cv_repeated(True,  False, "image-only")
oof_clin, _ = run_cv_repeated(False, True,  "clinical-only")
print()
for nm,o in [("clinical-only (logreg)",oof_lr),("clinical-only (deep)",oof_clin),
             ("image-only (deep)",oof_img),("FUSION",oof_fusion)]:
    print(f"{nm:24s} AUROC {roc_auc_score(labels.astype(int),o):.3f}  AUPRC {average_precision_score(labels.astype(int),o):.3f}")

[image-only] seed 0 fold 0 AUROC 0.589
[image-only] seed 0 fold 1 AUROC 0.458
[image-only] seed 0 fold 2 AUROC 0.361
[image-only] seed 0 fold 3 AUROC 0.597
[image-only] seed 0 fold 4 AUROC 0.590
[image-only] seed 0 mean fold AUROC 0.519
[image-only] OVERALL 1x5 folds: 0.519 +/- 0.095

[clinical-only] seed 0 fold 0 AUROC 0.694
[clinical-only] seed 0 fold 1 AUROC 0.736
[clinical-only] seed 0 fold 2 AUROC 0.736
[clinical-only] seed 0 fold 3 AUROC 0.708
[clinical-only] seed 0 fold 4 AUROC 0.715
[clinical-only] seed 0 mean fold AUROC 0.718
[clinical-only] OVERALL 1x5 folds: 0.718 +/- 0.016


clinical-only (logreg)   AUROC 0.651  AUPRC 0.308
clinical-only (deep)     AUROC 0.557  AUPRC 0.257
image-only (deep)        AUROC 0.453  AUPRC 0.184
FUSION                   AUROC 0.473  AUPRC 0.181


## 12. Save + single-patient inference

In [12]:
all_idx = np.arange(len(cohort))
final_model = train_one(all_idx, all_idx[:max(2,len(all_idx)//5)], True, True)
torch.save(final_model.state_dict(), OUTPUT_DIR/"egfr_fusion_v3.pt"); print("saved")

@torch.no_grad()
def predict_patient(case_id):
    i = cohort.index[cohort["Case ID"]==case_id][0]
    img = torch.from_numpy(np.load(CACHE_DIR/f"{case_id}.npy")).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    tab = torch.from_numpy(tab_mat[i]).float().unsqueeze(0).to(DEVICE)
    final_model.eval()
    return {"Case ID":case_id, "P(EGFR-mutant)":round(torch.sigmoid(final_model(img,tab)).item(),3),
            "true_label": POS_LABEL if labels[i]==1 else NEG_LABEL}
print(predict_patient(cohort.iloc[0]["Case ID"]))

saved
{'Case ID': 'R01-003', 'P(EGFR-mutant)': 0.601, 'true_label': 'Mutant'}


## 13. (OPTIONAL, reference only) AIM semantic-feature ceiling

Reference only — NOT part of the model. NB: in your last run this parsed 0/111. The *matching* is fine
(R01-003 finds its XML); the regex below just does not match your AIM schema's actual element/attribute
names. To fix, open one XML and print its tags, then adjust the patterns. Low priority.

In [14]:
import xml.etree.ElementTree as ET

def _local(tag):
    return tag.rsplit('}', 1)[-1]                 # strip XML namespace -> local tag name

def parse_aim(xml_path):
    """Namespace-agnostic AIM v4 parser -> {feature_label: value}."""
    feats = {}
    try:
        root = ET.parse(xml_path).getroot()
    except Exception:
        return feats
    for el in root.iter():
        if _local(el.tag) in ("ImagingObservationCharacteristic", "ImagingPhysicalEntity"):
            label = value = None
            for child in el:
                clt = _local(child.tag)
                if clt == "label":
                    label = child.get("value")
                elif clt == "typeCode" and value is None:
                    cs = (child.get("codeSystem") or "").strip()      # codeSystem holds the human value
                    if cs and "radlex" not in cs.lower():
                        value = cs
            if label and value:
                feats[label.strip()] = value.strip()
    return feats

if AIM_DIR.exists():
    all_xml = glob.glob(os.path.join(str(AIM_DIR), "*.xml"))
    rows, n_matched, first_fail = {}, 0, None
    for cid in cohort["Case ID"]:
        mf = [f for f in all_xml if cid in os.path.basename(f)]
        if not mf:
            continue
        n_matched += 1
        res = parse_aim(mf[0])
        if res:
            rows[cid] = res
        elif first_fail is None:
            first_fail = mf[0]
    print(f"Files in AIM_DIR: {len(all_xml)} | cohort matched: {n_matched}/{len(cohort)} | parsed: {len(rows)}")
    if first_fail:                                 # if anything matched but didn't parse, show it
        print("First matched-but-unparsed file:", os.path.basename(first_fail))
        print(open(first_fail, encoding='utf-8', errors='ignore').read()[:800])
    if rows:
        aim_df = pd.DataFrame.from_dict(rows, orient="index")
        print("AIM feature columns:", list(aim_df.columns))
        aim_oh = pd.get_dummies(aim_df.astype(str)).reindex(cohort["Case ID"]).fillna(0).values.astype(float)
        lr2 = make_pipeline(StandardScaler(with_mean=False),
                            LogisticRegression(max_iter=2000, class_weight="balanced"))
        oof_aim = cross_val_predict(lr2, aim_oh, labels.astype(int), cv=N_FOLDS, method="predict_proba")[:, 1]
        print(f"\nAIM semantic ceiling -> AUROC {roc_auc_score(labels.astype(int), oof_aim):.3f} "
              f"| AUPRC {average_precision_score(labels.astype(int), oof_aim):.3f} | cols {aim_oh.shape[1]}")
else:
    print(f"AIM_DIR not found at {AIM_DIR}")

Files in AIM_DIR: 190 | cohort matched: 110/111 | parsed: 0
First matched-but-unparsed file: R01-003.xml
<?xml version="1.0" encoding="UTF-8"?>
<ImageAnnotationCollection aimVersion="AIMv4_0" xmlns="gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM AIM_v4_rv44_XML.xsd">
<uniqueIdentifier root="wymknfz9qsc7gm4n6b7530m0adoxfgdrae3542oh"></uniqueIdentifier>
<dateTime value="20140502114854"></dateTime>
<user>
<name value="anonymized"></name>
<loginName value="anonymized"></loginName>
<roleInTrial></roleInTrial>
</user>
<equipment>
<manufacturerName value="GE MEDICAL SYSTEMS"></manufacturerName>
<manufacturerModelName value="Discovery CT750 HD"></manufacturerModelName>
<softwareVersion value="gsi_mict_plus.32"


## 14. Reading the results

- **Image-only:** did the slice-transformer beat v2's image-only (~0.64-0.66)? If yes, attending across the
  whole tumour depth helped. If it ties/trails, the EGFR signal in this small cohort's images is simply weak
  — an honest finding, not a bug.
- **Fusion vs clinical-only:** the headline. Judge it on **mean+/-std across repeats**, not a lucky fold.
- **If it overfits** (train AUROC high, val low / unstable): lower `VIT_LAYERS` to 1, `VIT_D_MODEL` to 128,
  raise `DROPOUT`, or pool fewer slices. Do **not** unfreeze the backbone at this N.
- **Bigger picture:** the highest-value next move is still recovering the ~61 AIM-coordinate patients
  (111 -> ~150-170, ~21 -> ~38 positives) — that shrinks the error bars more than any architecture change.